# 44 -- Start vs end weights and free energy for the full hyperbolic RBM (1k x 1k)

No training happens in this notebook. It loads the weights already trained and saved by
notebook 43 (`43_full_hyperbolic_rbm_1k_weights.npz`: `W_plus, c_plus, d_plus, W_minus, c_minus, d_minus`),
and separately rebuilds the untrained starting weights with the exact same initialization
code and seed (44) that notebook 43 used for the real-RBM draw. Both are compared on:

- mean |W+|, mean |W-|, mean |B| where B = (W+ - W-)/2
- the mean free energy of the real training rows, per channel:
  F(v) = -v^T d - sum_j log(1 + exp(c_j + (vW)_j))

using V+ = a (dot) M with (W+, c+, d+), and V- = b (dot) M with (W-, c-, d-).

## Setup -- data loading and matrix construction (copied from notebook 43, unchanged)

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
data_dir = root / "data"
out_dir = root / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

train_hyp = pd.read_csv(data_dir / "train_hyperbolic.csv")
test_hyp = pd.read_csv(data_dir / "test_hyperbolic.csv")
print(f"train_hyperbolic: {len(train_hyp):,} rows   test_hyperbolic: {len(test_hyp):,} rows")

EXPECTED_TRAIN, EXPECTED_TEST = 492055, 123474
if len(train_hyp) != EXPECTED_TRAIN or len(test_hyp) != EXPECTED_TEST:
    raise SystemExit(
        f"STOP: row counts differ from spec. expected train={EXPECTED_TRAIN:,} test={EXPECTED_TEST:,}, "
        f"got train={len(train_hyp):,} test={len(test_hyp):,}"
    )
print(f"confirmed: {len(train_hyp):,} training ratings, {len(test_hyp):,} test ratings")

all_users = sorted(pd.concat([train_hyp["userId"], test_hyp["userId"]]).unique())
all_movies = sorted(pd.concat([train_hyp["movieId"], test_hyp["movieId"]]).unique())
n_users, n_movies = len(all_users), len(all_movies)
user_to_row = {u: i for i, u in enumerate(all_users)}
movie_to_col = {m: j for j, m in enumerate(all_movies)}
print(f"n_users={n_users}  n_movies={n_movies}")

train_hyperbolic: 492,055 rows   test_hyperbolic: 123,474 rows
confirmed: 492,055 training ratings, 123,474 test ratings
n_users=1000  n_movies=1000


In [2]:
class RBM:
    def __init__(self, n_visible, n_hidden, seed=42, device="cpu"):
        g = torch.Generator(device=device).manual_seed(seed)
        self.n_visible = n_visible
        self.n_hidden = n_hidden
        self.device = device
        self.W = (torch.randn(n_visible, n_hidden, generator=g, device=device) * 0.01)
        self.vbias = torch.zeros(n_visible, device=device)
        self.hbias = torch.zeros(n_hidden, device=device)


def build_matrix(df, value_col):
    mat = torch.zeros(n_users, n_movies, dtype=torch.float32)
    mask = torch.zeros(n_users, n_movies, dtype=torch.float32)
    rows = torch.tensor(df["userId"].map(user_to_row).values, dtype=torch.long)
    cols = torch.tensor(df["movieId"].map(movie_to_col).values, dtype=torch.long)
    vals = torch.tensor(df[value_col].values, dtype=torch.float32)
    mat[rows, cols] = vals
    mask[rows, cols] = 1.0
    return mat, mask

## Visible input -- a, M, b, V_plus, V_minus (same construction as notebook 43 step 2)

In [3]:
N_HIDDEN = 128

train_hyp = train_hyp.copy()
train_hyp["a_new"] = (train_hyp["rating"] - 0.5) / 4.5

V_plus_train, mask_train = build_matrix(train_hyp, "a_new")
print(f"V_plus_train: shape={tuple(V_plus_train.shape)}  observed cells={int(mask_train.sum())}")

mu_u_series = train_hyp.groupby("userId")["rating"].mean().reindex(all_users)
if mu_u_series.isna().any():
    missing = mu_u_series[mu_u_series.isna()].index.tolist()
    raise SystemExit(f"STOP: {len(missing)} users have no training ratings: {missing[:10]}")
mu_u = torch.tensor(mu_u_series.values, dtype=torch.float32)
b_u = (mu_u - 0.5) / 4.5

V_minus_train = mask_train * b_u.unsqueeze(1)
print(f"V_minus_train: shape={tuple(V_minus_train.shape)}  observed cells={int((V_minus_train != 0).sum())}")

V_plus_train: shape=(1000, 1000)  observed cells=492055
V_minus_train: shape=(1000, 1000)  observed cells=492055


## Starting weights -- same init code and seed (44) as the R-baseline draw

In [4]:
SEED_WEIGHTS_INIT = 44

rbm_plus_start = RBM(n_visible=n_movies, n_hidden=N_HIDDEN, seed=SEED_WEIGHTS_INIT)
rbm_minus_start = RBM(n_visible=n_movies, n_hidden=N_HIDDEN, seed=SEED_WEIGHTS_INIT)

max_abs_diff_start = (rbm_plus_start.W - rbm_minus_start.W).abs().max().item()
print(f"max |W_plus - W_minus| at start: {max_abs_diff_start}")
assert max_abs_diff_start == 0.0, "STOP: start W_plus and W_minus are not identical"
print("confirmed: start W_plus == start W_minus (same seed, same draw).")

max |W_plus - W_minus| at start: 0.0
confirmed: start W_plus == start W_minus (same seed, same draw).


## Trained weights -- loaded from notebook 43's saved npz, not retrained

In [5]:
weights_path = out_dir / "43_full_hyperbolic_rbm_1k_weights.npz"
loaded = np.load(weights_path)

W_plus_end = torch.tensor(loaded["W_plus"])
c_plus_end = torch.tensor(loaded["c_plus"])
d_plus_end = torch.tensor(loaded["d_plus"])
W_minus_end = torch.tensor(loaded["W_minus"])
c_minus_end = torch.tensor(loaded["c_minus"])
d_minus_end = torch.tensor(loaded["d_minus"])

print(f"loaded: {weights_path.name}")
print(f"W_plus_end shape={tuple(W_plus_end.shape)}  W_minus_end shape={tuple(W_minus_end.shape)}")

loaded: 43_full_hyperbolic_rbm_1k_weights.npz
W_plus_end shape=(1000, 128)  W_minus_end shape=(1000, 128)


## Mean |W+|, mean |W-|, mean |B| -- start and end

In [6]:
def mean_abs_B(W_plus, W_minus):
    return ((W_plus - W_minus) / 2).abs().mean().item()

mean_abs_Wplus_start = rbm_plus_start.W.abs().mean().item()
mean_abs_Wminus_start = rbm_minus_start.W.abs().mean().item()
mean_abs_B_start = mean_abs_B(rbm_plus_start.W, rbm_minus_start.W)

mean_abs_Wplus_end = W_plus_end.abs().mean().item()
mean_abs_Wminus_end = W_minus_end.abs().mean().item()
mean_abs_B_end = mean_abs_B(W_plus_end, W_minus_end)

print(f"mean |W+|  start={mean_abs_Wplus_start:.6f}  end={mean_abs_Wplus_end:.6f}")
print(f"mean |W-|  start={mean_abs_Wminus_start:.6f}  end={mean_abs_Wminus_end:.6f}")
print(f"mean |B|   start={mean_abs_B_start:.6f}  end={mean_abs_B_end:.6f}")

mean |W+|  start=0.007978  end=0.015932
mean |W-|  start=0.007978  end=0.009396
mean |B|   start=0.000000  end=0.005537


## Free energy -- mean over the 1k users of the real training rows, per channel

In [7]:
def mean_free_energy(V, W, c, d):
    """F(v) = -v^T d - sum_j log(1 + exp(c_j + (vW)_j)), averaged over the rows of V."""
    term1 = -(V @ d)
    vW = V @ W
    term2 = -F.softplus(c + vW).sum(dim=1)
    return (term1 + term2).mean().item()

fe_plus_start = mean_free_energy(V_plus_train, rbm_plus_start.W, rbm_plus_start.hbias, rbm_plus_start.vbias)
fe_minus_start = mean_free_energy(V_minus_train, rbm_minus_start.W, rbm_minus_start.hbias, rbm_minus_start.vbias)

fe_plus_end = mean_free_energy(V_plus_train, W_plus_end, c_plus_end, d_plus_end)
fe_minus_end = mean_free_energy(V_minus_train, W_minus_end, c_minus_end, d_minus_end)

print(f"free energy, + channel:  start={fe_plus_start:.4f}  end={fe_plus_end:.4f}")
print(f"free energy, - channel:  start={fe_minus_start:.4f}  end={fe_minus_end:.4f}")

free energy, + channel:  start=-88.4767  end=-298.1158
free energy, - channel:  start=-88.5557  end=-267.7888


## Summary table

In [8]:
rows = [
    ("mean |W+|", mean_abs_Wplus_start, mean_abs_Wplus_end),
    ("mean |W-|", mean_abs_Wminus_start, mean_abs_Wminus_end),
    ("mean |B|", mean_abs_B_start, mean_abs_B_end),
    ("free energy, + channel", fe_plus_start, fe_plus_end),
    ("free energy, - channel", fe_minus_start, fe_minus_end),
]

print(f"{'':28s} {'Start':>14s} {'After 500 updates':>20s}")
for label, start_val, end_val in rows:
    print(f"{label:28s} {start_val:14.6f} {end_val:20.6f}")

                                      Start    After 500 updates
mean |W+|                          0.007978             0.015932
mean |W-|                          0.007978             0.009396
mean |B|                           0.000000             0.005537
free energy, + channel           -88.476669          -298.115753
free energy, - channel           -88.555672          -267.788818


## Save results

In [9]:
import json

results = {
    "note": "no training in this notebook; loads notebook 43's saved weights and rebuilds the start weights with the same seed (44)",
    "seed_weights_init": SEED_WEIGHTS_INIT,
    "n_users": n_users,
    "n_movies": n_movies,
    "n_hidden": N_HIDDEN,
    "mean_abs_W_plus": {"start": mean_abs_Wplus_start, "end": mean_abs_Wplus_end},
    "mean_abs_W_minus": {"start": mean_abs_Wminus_start, "end": mean_abs_Wminus_end},
    "mean_abs_B": {"start": mean_abs_B_start, "end": mean_abs_B_end},
    "free_energy_plus_channel": {"start": fe_plus_start, "end": fe_plus_end},
    "free_energy_minus_channel": {"start": fe_minus_start, "end": fe_minus_end},
}

results_path = out_dir / "44_start_vs_end_weights_free_energy_1k_results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"saved: {results_path}")

saved: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/outputs/44_start_vs_end_weights_free_energy_1k_results.json
